# Notebook 01 — Análisis Exploratorio de Datos

Exploración inicial de los datasets: distribución de clases, longitud de mensajes, vocabulario más frecuente y señales de fraude.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.data_loader import load_multiple_datasets, load_and_normalize
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

## 1. Carga de datos

In [ ]:
# Cargar todos los CSV de data/raw
try:
    df = load_multiple_datasets(RAW_DATA_DIR)
    print(f'Total de mensajes: {len(df)}')
    df.head()
except FileNotFoundError as e:
    print(f'No se encontraron datasets: {e}')
    print('Coloca tus archivos CSV en data/raw/ y vuelve a ejecutar.')
    # Dataset de demostración
    df = pd.DataFrame({
        'message': [
            'Your account is blocked. Verify at http://fake-bank.com',
            'Hi, how are you doing today?',
            'URGENT: You won $10,000! Send your card number now.',
            'Reminder: your appointment is tomorrow at 10am.',
            'Send your password and PIN to verify your account.',
        ],
        'label': ['fraudulent', 'legitimate', 'fraudulent', 'legitimate', 'fraudulent'],
        'source': ['demo'] * 5
    })
    print('Usando dataset de demostración.')
    df.head()

## 2. Distribución de clases

In [ ]:
label_counts = df['label'].value_counts()
print(label_counts)
print(f'\nBalance de clases (%):')
print((label_counts / len(df) * 100).round(2))

label_counts.plot(kind='bar', color=['steelblue', 'tomato', 'orange'], rot=0)
plt.title('Distribución de clases')
plt.ylabel('Cantidad de mensajes')
plt.tight_layout()
plt.show()

## 3. Longitud de mensajes por clase

In [ ]:
df['msg_length'] = df['message'].astype(str).str.len()
df['word_count'] = df['message'].astype(str).str.split().str.len()

print(df.groupby('label')[['msg_length', 'word_count']].describe().round(1))

for label in df['label'].unique():
    subset = df[df['label'] == label]['msg_length']
    plt.hist(subset, bins=40, alpha=0.6, label=label)
plt.title('Distribución de longitud de mensajes por clase')
plt.xlabel('Longitud (caracteres)')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Señales de fraude más frecuentes

In [ ]:
import re
from src.config import URGENCY_WORDS, CREDENTIAL_WORDS, PRIZE_WORDS, MONEY_WORDS

def signal_presence(df, word_list, signal_name):
    df[signal_name] = df['message'].apply(
        lambda t: int(any(w in str(t).lower() for w in word_list))
    )
    return df

df = signal_presence(df, URGENCY_WORDS, 'has_urgency')
df = signal_presence(df, CREDENTIAL_WORDS, 'has_credentials')
df = signal_presence(df, PRIZE_WORDS, 'has_prize')
df = signal_presence(df, MONEY_WORDS, 'has_money_words')
df['has_url'] = df['message'].str.contains(r'https?://|www\.', regex=True, case=False).astype(int)

signal_cols = ['has_urgency', 'has_credentials', 'has_prize', 'has_money_words', 'has_url']
signal_summary = df.groupby('label')[signal_cols].mean().round(3)
print('Proporción de señales por clase:')
print(signal_summary)

signal_summary.T.plot(kind='bar', rot=30)
plt.title('Presencia de señales por clase')
plt.ylabel('Proporción')
plt.tight_layout()
plt.show()

## 5. Palabras más frecuentes por clase

Completa esta sección con el análisis de n-gramas más frecuentes usando CountVectorizer.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

for label in ['fraudulent', 'legitimate']:
    subset = df[df['label'] == label]['message'].astype(str).tolist()
    if not subset:
        continue
    cv = CountVectorizer(max_features=15, stop_words=None, ngram_range=(1,1))
    cv.fit(subset)
    freqs = cv.transform(subset).sum(axis=0).A1
    words = cv.get_feature_names_out()
    top = sorted(zip(words, freqs), key=lambda x: -x[1])[:15]
    print(f'\nTop palabras — {label.upper()}:')
    for w, f in top:
        print(f'  {w:20s}: {f}')